# 04 Splits

Builds the random, lineage and time splits for each antibiotic and checks that they are correct.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys

src_dir = Path("../src").resolve()
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import splits
import data
ANTIBIOTICS = data.ANTIBIOTICS
X, labels, year, clusters = data.load_data()

main_antibiotics = ["CIP", "GEN", "CXM"]

## Random split

In [4]:
from sklearn.model_selection import StratifiedKFold

def random_split(antibiotic):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    folds = []
    y = labels[antibiotic].dropna()
    for train_pos, test_pos in skf.split(X.loc[y.index], y):
        train_ids = y.index[train_pos]
        test_ids = y.index[test_pos]
        folds.append((train_ids, test_ids))
    return folds
    

In [5]:
folds = random_split("GEN")
print("Number of folds:", len(folds))
train_ids, test_ids = folds[0]
print("First fold: train", len(train_ids), " test", len(test_ids))
print("Example test IDs:", list(test_ids[:3]))

Number of folds: 5
First fold: train 1548  test 387
Example test IDs: ['11657_5#10', '11657_5#13', '11657_5#17']


## Lineage split

In [7]:
from sklearn.model_selection import StratifiedGroupKFold

def lineage_split(antibiotic):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    folds = []
    y = labels[antibiotic].dropna()
    groups = clusters.loc[y.index, "cutoff_1168"]
    for train_pos, test_pos in sgkf.split(X.loc[y.index], y, groups):
        train_ids = y.index[train_pos]
        test_ids = y.index[test_pos]
        folds.append((train_ids, test_ids))
    return folds
    
    

In [8]:
folds = lineage_split("GEN")
print("Number of folds:", len(folds))
train_ids, test_ids = folds[0]
print("First fold: train", len(train_ids), " test", len(test_ids))
print("Example test IDs:", list(test_ids[:3]))

Number of folds: 5
First fold: train 1658  test 277
Example test IDs: ['11657_5#10', '11657_5#17', '11657_5#29']


In [9]:
for i, (train_ids, test_ids) in enumerate(folds):
    train_families = set(clusters.loc[train_ids, "cutoff_1168"])
    test_families = set(clusters.loc[test_ids, "cutoff_1168"])
    shared = train_families & test_families
    print(f"Fold {i + 1}: {len(test_families)} test families, {len(shared)} shared with train")

Fold 1: 97 test families, 0 shared with train
Fold 2: 131 test families, 0 shared with train
Fold 3: 130 test families, 0 shared with train
Fold 4: 134 test families, 0 shared with train
Fold 5: 131 test families, 0 shared with train


No family appears in both train and test in any lineage fold.

## Time split

In [12]:
def time_split(antibiotic):
    y = labels[antibiotic].dropna()
    y_year = year.loc[y.index]

    in_train = y_year.between(2001, 2008).fillna(False)
    in_test = y_year.between(2009, 2012).fillna(False)

    train_ids = in_train[in_train].index
    test_ids = in_test[in_test].index
    return [(train_ids, test_ids)]

In [13]:
train_ids, test_ids = time_split("GEN")[0]
print("Strains in both:", len(set(train_ids) & set(test_ids)))

Strains in both: 0


## Check all splits

In [15]:
rows = []
for antibiotic in main_antibiotics:
    all_splits = {
        "random": splits.random_split(labels, antibiotic),
        "lineage": splits.lineage_split(labels, clusters, antibiotic),
        "time": splits.time_split(labels, year, antibiotic),
    }
    for split_name, folds in all_splits.items():
        for i, (train_ids, test_ids) in enumerate(folds):
            test_labels = labels.loc[test_ids, antibiotic]
            rows.append({
                "antibiotic": antibiotic,
                "split": split_name,
                "fold": i + 1,
                "n_train": len(train_ids),
                "n_test": len(test_ids),
                "test_R": (test_labels == "R").sum(),
                "overlap": len(set(train_ids) & set(test_ids)),
            })

split_check = pd.DataFrame(rows)
split_check

,antibiotic,split,fold,n_train,n_test,test_R,overlap
0,CIP,random,1,1548,387,86,0
1,CIP,random,2,1548,387,86,0
2,CIP,random,3,1548,387,86,0
3,CIP,random,4,1548,387,86,0
4,CIP,random,5,1548,387,85,0
5,CIP,lineage,1,1686,249,41,0
6,CIP,lineage,2,1564,371,45,0
7,CIP,lineage,3,1513,422,41,0
8,CIP,lineage,4,1382,553,237,0
9,CIP,lineage,5,1595,340,65,0


In [16]:
split_check.groupby(["antibiotic", "split"])[["n_test", "test_R", "overlap"]].agg(["min", "max"])

n_test      test_R      overlap    
                      min  max    min  max     min max
antibiotic split                                      
CIP        lineage    249  553     41  237       0   0
           random     387  387     85   86       0   0
           time       624  624    125  125       0   0
CXM        lineage    188  530     43  210       0   0
           random     387  387    104  104       0   0
           time       624  624    130  130       0   0
GEN        lineage    277  521     35  110       0   0
           random     387  387     56   57       0   0
           time       624  624     54   54       0   0

No overlap in any split; every test fold has at least 35 resistant strains.

## Summary

- Three splits are built per antibiotic: random (5 folds), lineage (5 folds, cutoff 1,168) and time (train 2001–2008, test 2009–2012).
- No strain appears in both train and test in any split.
- No family appears in both train and test in any lineage fold.
- Every test fold has at least 54 resistant strains for each main antibiotic.
- The split functions are saved in `src/splits.py` for the model notebook.